In [1]:
import pandas as pd
import numpy as np

# =========================================
# LOAD DATA
# =========================================
path = "/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/data/CO2_to_DME_Data.xlsx - Sheet1.csv"
df = pd.read_csv(path)

print("\n===== BASIC INFO =====")
print("Shape:", df.shape)
display(df.head())


===== BASIC INFO =====
Shape: (656, 23)


,Integration_method,Reaction_temperature_C,Pressure_MPa,H2_CO2_ratio,GHSV_mL_gcat_h,catalyst_mass_g,metal_1_atomic,metal_2_atomic,metal_3_atomic,metal_4_atomic,...,acid_component_type,Dimentionality of Zeolite,Si/Al_ratio,Pore_volume_cm3_g,Total_acid_sites_mmol_g,Metal-acid_ratio,CO Selectivity (%)\t\t\t\t,DME Selectivity (%),Methanol Selectivity (%)\t\t\t\t,CO2 conversion efficiency (%)
0,Physical mixing,260,5.0,3.0,5000,0.4,29,30.0,13.0,NaN,...,FER,2D,11.7,0.34,0.56,1.79,50.0,43.2,6.7,18.7
1,Structured or Spatially Integrated Systems,260,5.0,3.0,5000,0.4,29,30.0,13.0,NaN,...,FER,2D,11.7,0.33,0.08,12.50,38.8,54.4,6.8,23.1
2,Physical mixing,260,5.0,3.0,5000,0.4,29,30.0,13.0,NaN,...,FER,2D,11.7,0.34,0.24,25.00,41.1,52.2,6.7,18.9
3,Physical mixing,260,5.0,3.0,5000,0.4,29,30.0,13.0,NaN,...,FER,2D,11.7,0.34,0.24,1.79,48.8,46.1,5.0,23.1
4,Physical mixing,260,5.0,3.0,5000,0.4,29,30.0,13.0,NaN,...,FER,NaN,NaN,0.30,0.00,NaN,32.9,0.0,67.1,23.4


In [2]:
# =========================================
# PANDAS PROFILING / EDA
# =========================================

print("\n===== COLUMN NAMES & DTYPES =====")
print(df.dtypes)

print("\n===== MISSING VALUES =====")
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Missing %': missing_pct.round(2)
})
print(missing_df[missing_df['Missing Count'] > 0].sort_values('Missing %', ascending=False))
print(f"\nTotal missing cells: {df.isnull().sum().sum()} / {df.size}")

print("\n===== DUPLICATES =====")
print(f"Duplicate rows: {df.duplicated().sum()}")

print("\n===== NUMERICAL SUMMARY =====")
display(df.describe(percentiles=[.05, .25, .5, .75, .95]).T.round(3))

print("\n===== CATEGORICAL SUMMARY =====")
cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    for col in cat_cols:
        print(f"\n  [{col}]  unique={df[col].nunique()}")
        print(df[col].value_counts().head(10).to_string())
else:
    print("No categorical columns found.")

print("\n===== NUMERICAL COLUMN STATS =====")
num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
stats_df = pd.DataFrame({
    'dtype':    df[num_cols].dtypes,
    'nunique':  df[num_cols].nunique(),
    'min':      df[num_cols].min(),
    'max':      df[num_cols].max(),
    'mean':     df[num_cols].mean().round(4),
    'median':   df[num_cols].median().round(4),
    'std':      df[num_cols].std().round(4),
    'skew':     df[num_cols].skew().round(4),
    'kurtosis': df[num_cols].kurt().round(4),
    'zeros':    (df[num_cols] == 0).sum(),
    'negatives':(df[num_cols] < 0).sum(),
})
display(stats_df)

print("\n===== CORRELATION MATRIX (Numerical) =====")
if len(num_cols) > 1:
    display(df[num_cols].corr().round(3))
else:
    print("Not enough numerical columns for correlation.")

print("\n===== OUTLIER DETECTION (IQR method) =====")
outlier_summary = []
for col in num_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    n_outliers = ((df[col] < lower) | (df[col] > upper)).sum()
    outlier_summary.append({'column': col, 'lower_fence': round(lower,4),
                            'upper_fence': round(upper,4), 'outlier_count': n_outliers,
                            'outlier_%': round(n_outliers / len(df) * 100, 2)})
display(pd.DataFrame(outlier_summary).set_index('column').sort_values('outlier_count', ascending=False))

print("\n===== PROFILING COMPLETE =====")


===== COLUMN NAMES & DTYPES =====
Integration_method                                         str
 Reaction_temperature_C                                  int64
Pressure_MPa                                           float64
H2_CO2_ratio                                           float64
GHSV_mL_gcat_h                                             str
catalyst_mass_g                                        float64
metal_1_atomic                                           int64
metal_2_atomic                                         float64
metal_3_atomic                                         float64
metal_4_atomic                                         float64
Cu_Metal_atomic_fraction                               float64
Surface_area_m2_g                                      float64
Crystallite_size_nm                                    float64
acid_component_type                                        str
Dimentionality of Zeolite                                  str
Si/Al_ratio         

,count,mean,std,min,5%,25%,50%,75%,95%,max
Reaction_temperature_C,656.0,258.091,32.396,150.000,220.000,240.000,260.000,270.000,300.000,400.00
Pressure_MPa,656.0,3.337,1.106,0.100,2.000,3.000,3.000,4.000,5.000,6.50
H2_CO2_ratio,656.0,3.178,1.637,1.000,2.000,3.000,3.000,3.000,4.000,16.50
catalyst_mass_g,538.0,0.840,0.769,0.020,0.100,0.250,0.600,1.000,2.660,4.50
metal_1_atomic,656.0,29.363,2.459,29.000,29.000,29.000,29.000,29.000,29.000,46.00
metal_2_atomic,645.0,30.090,5.376,13.000,22.000,30.000,30.000,30.000,40.000,49.00
metal_3_atomic,523.0,30.107,15.038,8.000,13.000,13.000,40.000,40.000,40.000,78.00
metal_4_atomic,91.0,25.143,19.308,8.000,12.000,13.000,13.000,31.000,74.000,74.00
Cu_Metal_atomic_fraction,468.0,0.495,0.181,0.000,0.075,0.460,0.534,0.600,0.670,1.00
Surface_area_m2_g,607.0,173.206,100.644,5.000,51.000,106.610,164.200,222.500,370.000,806.00



===== CATEGORICAL SUMMARY =====

  [Integration_method]  unique=27
Integration_method
Physical mixing             372
Coprecipitation             123
Dualbed                      29
In-situ introduction         20
Sequential impregnation      20
Sequential precipitation     14
Single-bed                   10
Physical coating              9
Wet impregnation              7
Core–shell capsule            6

  [GHSV_mL_gcat_h]  unique=57
GHSV_mL_gcat_h
8800     76
1500     56
6000     53
3600     44
3000     33
1800     28
2400     28
4200     21
10000    20
9000     18

  [acid_component_type]  unique=22
acid_component_type
HZSM-5     307
FER         96
Al2O3       69
SAPO-34     43
SAPO-18     19
SAPO-11     13
FAU         11
SBA-15      10
K10         10
HPW-K10     10

  [Dimentionality of Zeolite]  unique=4
Dimentionality of Zeolite
3D     400
2D     138
1D      23
2D       1

  [Metal-acid_ratio]  unique=73
Metal-acid_ratio
1        119
100       85
2         51
2:01      30
0.5     

/var/folders/m0/48s55r5n5gdbgpx0y7839_840000gn/T/ipykernel_84147/3932572557.py:25: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()


,dtype,nunique,min,max,mean,median,std,skew,kurtosis,zeros,negatives
Reaction_temperature_C,int64,22,150.0000,400.00,258.0915,260.000,32.3961,1.7874,7.3153,0,0
Pressure_MPa,float64,19,0.1000,6.50,3.3375,3.000,1.1058,-0.1513,0.9639,0,0
H2_CO2_ratio,float64,14,1.0000,16.50,3.1775,3.000,1.6367,7.0927,54.0312,0,0
catalyst_mass_g,float64,27,0.0200,4.50,0.8397,0.600,0.7694,2.0962,5.2740,0,0
metal_1_atomic,int64,2,29.0000,46.00,29.3628,29.000,2.4587,6.6393,42.2091,0,0
metal_2_atomic,float64,9,13.0000,49.00,30.0899,30.000,5.3760,-0.0997,5.8339,0,0
metal_3_atomic,float64,11,8.0000,78.00,30.1071,40.000,15.0376,0.2431,-0.2574,0,0
metal_4_atomic,float64,13,8.0000,74.00,25.1429,13.000,19.3084,1.5507,1.2993,0,0
Cu_Metal_atomic_fraction,float64,43,0.0000,1.00,0.4945,0.534,0.1806,-1.0323,1.3997,14,0
Surface_area_m2_g,float64,232,5.0000,806.00,173.2057,164.200,100.6436,1.3563,3.7143,0,0



===== CORRELATION MATRIX (Numerical) =====


,Reaction_temperature_C,Pressure_MPa,H2_CO2_ratio,catalyst_mass_g,metal_1_atomic,metal_2_atomic,metal_3_atomic,metal_4_atomic,Cu_Metal_atomic_fraction,Surface_area_m2_g,Crystallite_size_nm,Si/Al_ratio,Pore_volume_cm3_g,Total_acid_sites_mmol_g,CO Selectivity (%)\t\t\t\t,DME Selectivity (%),Methanol Selectivity (%)\t\t\t\t,CO2 conversion efficiency (%)
Reaction_temperature_C,1.000,-0.277,-0.009,-0.109,0.178,-0.361,-0.265,-0.009,-0.530,-0.063,0.334,0.009,0.169,0.023,0.065,0.237,-0.043,0.418
Pressure_MPa,-0.277,1.000,0.025,0.192,-0.055,0.299,-0.125,0.044,0.221,0.129,-0.031,-0.002,-0.054,-0.137,-0.164,-0.047,0.090,-0.094
H2_CO2_ratio,-0.009,0.025,1.000,-0.040,-0.016,-0.038,-0.053,-0.387,0.102,0.214,-0.053,-0.002,-0.050,-0.008,-0.164,0.093,-0.072,0.354
catalyst_mass_g,-0.109,0.192,-0.040,1.000,-0.138,0.017,-0.344,-0.078,0.015,0.078,-0.128,-0.055,-0.114,0.507,-0.111,-0.004,-0.208,0.070
metal_1_atomic,0.178,-0.055,-0.016,-0.138,1.000,-0.002,NaN,NaN,-0.481,0.041,0.820,-0.005,0.758,-0.017,0.091,-0.155,0.072,-0.087
metal_2_atomic,-0.361,0.299,-0.038,0.017,-0.002,1.000,0.207,0.168,0.034,0.134,0.020,-0.028,-0.086,-0.016,0.177,-0.246,0.075,-0.372
metal_3_atomic,-0.265,-0.125,-0.053,-0.344,NaN,0.207,1.000,-0.219,0.075,0.060,-0.160,0.028,-0.181,-0.058,0.172,-0.078,-0.076,-0.435
metal_4_atomic,-0.009,0.044,-0.387,-0.078,NaN,0.168,-0.219,1.000,-0.004,-0.315,0.275,-0.109,-0.164,-0.401,0.649,0.011,-0.072,0.079
Cu_Metal_atomic_fraction,-0.530,0.221,0.102,0.015,-0.481,0.034,0.075,-0.004,1.000,-0.145,-0.489,0.122,-0.338,-0.140,0.019,-0.174,-0.023,-0.063
Surface_area_m2_g,-0.063,0.129,0.214,0.078,0.041,0.134,0.060,-0.315,-0.145,1.000,0.013,-0.040,0.024,-0.011,-0.038,0.146,-0.186,0.066



===== OUTLIER DETECTION (IQR method) =====


,lower_fence,upper_fence,outlier_count,outlier_%
column,,,,
metal_2_atomic,30.0000,30.0000,114,17.38
H2_CO2_ratio,3.0000,3.0000,96,14.63
Cu_Metal_atomic_fraction,0.2500,0.8100,62,9.45
Methanol Selectivity (%)\t\t\t\t,-19.2750,48.1250,56,8.54
CO2 conversion efficiency (%),-15.6375,49.2625,48,7.32
Crystallite_size_nm,-11.2500,38.7500,46,7.01
Pore_volume_cm3_g,-0.1585,0.6935,42,6.40
catalyst_mass_g,-0.8750,2.1250,38,5.79
Pressure_MPa,1.5000,5.5000,33,5.03



===== PROFILING COMPLETE =====


In [4]:
!pip install ydata-profiling

ERROR: Ignored the following versions that require a different python version: 4.0.0 Requires-Python >=3.7,<3.11; 4.1.0 Requires-Python >=3.7,<3.12; 4.1.1 Requires-Python >=3.7,<3.12; 4.1.2 Requires-Python >=3.7,<3.12; 4.10.0 Requires-Python >=3.7,<3.13; 4.11.0 Requires-Python >=3.7,<3.13; 4.12.0 Requires-Python >=3.7,<3.13; 4.12.1 Requires-Python >=3.7,<3.13; 4.12.2 Requires-Python >=3.7,<3.13; 4.13.0 Requires-Python >=3.7,<3.13; 4.14.0 Requires-Python >=3.7,<3.13; 4.15.0 Requires-Python >=3.7,<3.13; 4.15.1 Requires-Python >=3.7,<3.13; 4.16.0 Requires-Python >=3.7,<3.13; 4.16.1 Requires-Python >=3.7,<3.13; 4.17.0 Requires-Python >=3.7,<3.14; 4.18.0 Requires-Python >=3.10,<3.14; 4.18.1 Requires-Python >=3.10,<3.14; 4.2.0 Requires-Python >=3.7,<3.12; 4.3.0 Requires-Python >=3.7,<3.12; 4.3.1 Requires-Python >=3.7,<3.12; 4.3.2 Requires-Python >=3.7,<3.12; 4.4.0 Requires-Python >=3.7,<3.12; 4.5.0 Requires-Python >=3.7,<3.12; 4.5.1 Requires-Python >=3.7,<3.12; 4.6.0 Requires-Python >=3.7,<3

In [ ]:
# =========================================
# PROFILE REPORT (sweetviz - works on Python 3.14+)
# =========================================
!pip install sweetviz

import sweetviz as sv

report = sv.analyze(df)
report.show_html("output(before impute).html")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.1/15.1 MB 7.4 MB/s  0:00:02 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [sweetviz]


                                             |          | [  0%]   00:00 -> (? left)

/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/.venv/lib/python3.14/site-packages/sweetviz/graph.py:34: UserWarning: Glyph 8323 (\N{SUBSCRIPT THREE}) missing from font(s) Roboto.
  figure.savefig(as_raw_bytes, format="png", transparent=True)
/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/.venv/lib/python3.14/site-packages/sweetviz/graph.py:34: UserWarning: Glyph 8322 (\N{SUBSCRIPT TWO}) missing from font(s) Roboto.
  figure.savefig(as_raw_bytes, format="png", transparent=True)
/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/.venv/lib/python3.14/site-packages/sweetviz/graph.py:34: UserWarning: Glyph 8325 (\N{SUBSCRIPT FIVE}) missing from font(s) Roboto.
  figure.savefig(as_raw_bytes, format="png", transparent=True)
/Users/kanishq/Documents/Projects/CO2_to_DME/vs_code/.venv/lib/python3.14/site-packages/sweetviz/graph.py:34: UserWarning: Glyph 9 (	) missing from font(s) Roboto.
  figure.savefig(as_raw_bytes, format="png", transparent=True)


Report output.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.
